# SPIN Healthcare — Milestone 2, Step 2: decoupling (HSIC) + capability (perplexity)

Measures what suppressing the healthcare coupled set does to (a) the **fairness↔privacy representation
dependence** (HSIC — the decoupling claim, Proposition 1) and (b) **capability** (perplexity). Baseline
(no suppression) is measured once; the coupled set is zeroed, re-measured, then restored (reversible, no
reload). HSIC estimator / σ grid / raw-question protocol / ppl window are **identical** to the
reproduction's `SPIN_iter_step2.ipynb`; only the eval data changes (healthcare) and the stat suite is
extended.

**This notebook also builds the eval splits** (held-out, disjoint from Step-1 calibration) and saves them
for Steps 3–4: `eval_fair_health.csv` (question + subset, all held-out EquityMedQA questions across the 7
subsets) and `eval_priv_health.csv` (657 MedSafetyBench cat_4 questions, 2 calibration overlaps removed).

**Full stat suite extracted here**
- **Raw HSIC** (paper-faithful, unnormalised) final-layer + **per-layer**, at σ∈{50,100,200,400}, baseline vs SPIN, Δ abs & %.
- **Normalised HSIC** (nHSIC = HSIC(X,Y)/√(HSIC(X,X)·HSIC(Y,Y))) and **linear CKA** — norm-robust decoupling checks (the raw-HSIC-vs-norm confound noted in the Iter-SPIN caveats).
- **Representation-norm diagnostics** (mean ‖·‖ of fairness/privacy reps, baseline vs SPIN) — to attribute any HSIC change to dependence vs norm shrink.
- **Bootstrap 95% CIs** on the HSIC drop (paired resampling), per σ.
- **Capability:** WikiText-2 perplexity (paper) **and** PubMedQA medical-text perplexity (domain-relevant capability retention), baseline vs SPIN.
- **Per-subset HSIC** (exploratory): HSIC(each fairness subset ↔ privacy) baseline vs SPIN — decoupling resolved by bias type (Ikram's per-subset angle at the representation level). Small subsets flagged (wide/noisy).
- **Suppression accounting:** weights zeroed, intervention sparsity.

**Inputs on Kaggle:** Step-1 output as a Dataset input (needs `cal_fair_health.csv`, `cal_priv_health.csv`,
and coupled indices — `coupled_indices_health.json` or `health_coupled_ckpt.pt`), plus
`equitymedqa_ratings.xlsx`. Internet ON (clones MedSafetyBench, pulls PubMedQA/WikiText-2 + the model).
GPU **T4×2**. **Output:** `metrics_health_step2.json` + the two eval CSVs (+ `coupled_indices_health.json`).

## Step 0 — config

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
import subprocess, sys, glob, re, gc, time, json
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.46.3','datasets==2.18.0',
                'sentencepiece','openpyxl'], check=False)
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

MODEL_ID='Qwen/Qwen2-7B-Instruct'          # must match the model whose coupled set you're scoring
# MODEL_ID='Qwen/Qwen2.5-0.5B-Instruct'
USE_FAST=False
SEED=0
SIGMAS=[50,100,200,400]                    # authors' σ grid (raw, final-layer, unnormalised = paper fidelity)
HSIC_N=300                                 # reps per side for the pooled HSIC (reproduction default)
PER_SUB_CAP=150                            # cap per subset for the per-subset HSIC pool (memory/time)
N_PRIV_EVAL=657                            # match the reproduction's privacy eval scale
PPL_SEQLEN=2048                            # perplexity window (memory-bounded, same as reproduction)
BOOT=500                                   # bootstrap resamples for HSIC 95% CIs (0 to skip)
MED_PPL=True                               # also measure PubMedQA medical-text perplexity
USE_CHAT_TEMPLATE=False                    # raw-question protocol (authors' eval_salad)
DTYPE=torch.float16
SUBSET_ORDER=['OMAQ','EHAI','FBRT-Manual','FBRT-LLM','TRINDS','CC-Manual','CC-LLM']
MODEL_TAG=MODEL_ID.split('/')[-1]
np.random.seed(SEED); torch.manual_seed(SEED)
def memline(tag=''):
    s=''
    for d in range(torch.cuda.device_count()):
        f,t=torch.cuda.mem_get_info(d); s+=f' GPU{d} {f/1e9:.1f}/{t/1e9:.1f}'
    print(f'  [{tag}]{s} GiB free')
def find_one(names):
    roots=['.','/kaggle/input','/kaggle/working']
    for nm in names:
        for r in roots:
            for f in glob.glob(os.path.join(r,'**',nm),recursive=True):
                return f
    return None
print('config ready | model', MODEL_ID, '| σ', SIGMAS, '| HSIC_N', HSIC_N)

## Step 1 — build eval splits (held-out, disjoint from Step-1 calibration)

### 1a — fairness eval: held-out EquityMedQA questions, per subset

In [ ]:
EQUITY_XLSX=find_one(['equitymedqa_ratings.xlsx'])
assert EQUITY_XLSX, 'Upload equitymedqa_ratings.xlsx as a Kaggle Dataset input.'
CAL_FAIR=find_one(['cal_fair_health.csv']); assert CAL_FAIR, 'Add Step-1 output (cal_fair_health.csv) as input.'

raw=pd.read_excel(EQUITY_XLSX, sheet_name='Independent ratings')
def subset_of(name):
    for t in SUBSET_ORDER:
        if f'({t})' in str(name): return t
    return None
raw=raw.dropna(subset=['question_text']).copy(); raw['subset']=raw['dataset'].apply(subset_of)
df=raw[raw['subset'].notna()].copy()
# unique (subset, question); exclude the exact calibration questions actually drawn in Step 1
cal_q=set(pd.read_csv(CAL_FAIR)['prompt'].astype(str))
uq=df.drop_duplicates(['subset','question_text'])[['subset','question_text']]
eval_fair=uq[~uq['question_text'].astype(str).isin(cal_q)].rename(columns={'question_text':'question'}).reset_index(drop=True)
eval_fair.to_csv('eval_fair_health.csv',index=False)
print('fairness eval (held-out) per subset:')
print(eval_fair['subset'].value_counts().reindex(SUBSET_ORDER).to_string())
print('total held-out fairness questions:', len(eval_fair))

### 1b — privacy eval: MedSafetyBench cat_4 questions (657, calibration overlaps removed)

In [ ]:
MEDSAFE_DIR='/tmp/med-safety-bench'    # clone OUTSIDE /kaggle/working so it isn't saved as output
if not os.path.exists(MEDSAFE_DIR):
    subprocess.run(['git','clone','--depth','1','https://github.com/AI4LIFE-GROUP/med-safety-bench.git', MEDSAFE_DIR], check=True)
CAT4=f'{MEDSAFE_DIR}/datasets/med_harm_llama3/category_4.txt'
q_all=[l.strip() for l in open(CAT4,encoding='utf-8',errors='replace') if l.strip()]
print('category_4 privacy-violation questions available:', len(q_all))

CAL_PRIV=find_one(['cal_priv_health.csv']); assert CAL_PRIV, 'Add Step-1 output (cal_priv_health.csv) as input.'
def norm(s): return re.sub(r'\s+',' ',str(s).strip().lower())
cal_norm=set(norm(x) for x in pd.read_csv(CAL_PRIV)['prompt'])
q_disjoint=[q for q in q_all if norm(q) not in cal_norm]
removed=len(q_all)-len(q_disjoint)
rng=np.random.default_rng(SEED)
idx=rng.permutation(len(q_disjoint))[:N_PRIV_EVAL]
eval_priv=pd.DataFrame({'question':[q_disjoint[i] for i in sorted(idx)]})
eval_priv.to_csv('eval_priv_health.csv',index=False)
print(f'removed {removed} calibration-overlapping question(s); sampled {len(eval_priv)} privacy eval questions')

## Step 2 — model, HSIC question pools, WikiText-2 + PubMedQA capability text

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID, use_fast=USE_FAST, trust_remote_code=True)
tokenizer.padding_side='left'
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
model=AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, trust_remote_code=True,
        device_map='auto', max_memory={0:'14GiB',1:'14GiB','cpu':'28GiB'}).eval()
GEN_DEV=model.get_input_embeddings().weight.device
print('model loaded | entry', GEN_DEV); memline('loaded')

# pooled HSIC question sets: fairness sampled STRATIFIED across subsets (so the pooled fairness reps
# represent all 7 bias types, not just the largest subset), privacy sampled from the 657 eval set.
def stratified_sample(dfq, n, seed=SEED):
    subs=dfq['subset'].unique(); per=max(1,n//len(subs)); rng=np.random.default_rng(seed); parts=[]
    for s in subs:
        d=dfq[dfq['subset']==s]; parts.append(d.sample(n=min(per,len(d)),random_state=seed))
    out=pd.concat(parts)
    if len(out)>n: out=out.sample(n=n,random_state=seed)
    return out
fair_pooled=stratified_sample(eval_fair, HSIC_N)['question'].tolist()
priv_pooled=eval_priv['question'].sample(n=min(HSIC_N,len(eval_priv)),random_state=SEED).tolist()
# per-subset pools (final-layer HSIC), capped
fair_by_sub={s:eval_fair[eval_fair['subset']==s]['question'].tolist()[:PER_SUB_CAP] for s in SUBSET_ORDER}
priv_sub=eval_priv['question'].tolist()[:PER_SUB_CAP]
print('pooled HSIC | fair', len(fair_pooled), '| priv', len(priv_pooled))

# capability corpora
_wt=load_dataset('wikitext','wikitext-2-raw-v1',split='test')['text']
wt_ids=tokenizer('\n\n'.join(_wt), return_tensors='pt').input_ids
med_ids=None
if MED_PPL:
    try: pq=load_dataset('pubmed_qa','pqa_labeled',split='train')
    except Exception: pq=load_dataset('pubmed_qa','pqa_labeled',split='train',trust_remote_code=True)
    cal_g=find_one(['cal_general_health.csv'])
    used=set(pd.read_csv(cal_g)['prompt'].astype(str)) if cal_g else set()
    med_text=[str(a) for q,a in zip(pq['question'],pq['long_answer']) if str(q) not in used]
    med_ids=tokenizer('\n\n'.join(med_text), return_tensors='pt').input_ids
    print('medical ppl corpus tokens:', med_ids.numel())
print('wikitext tokens:', wt_ids.numel())

## Step 3 — metric functions (HSIC/nHSIC/CKA/norms/ppl mirror the reproduction)

In [ ]:
def fmt(chunk):
    if USE_CHAT_TEMPLATE:
        return [tokenizer.apply_chat_template([{'role':'user','content':q}],tokenize=False,add_generation_prompt=True) for q in chunk]
    return [str(q) for q in chunk]
@torch.no_grad()
def reps_all(queries,bs=8):
    acc=None
    for i in range(0,len(queries),bs):
        enc=tokenizer(fmt(queries[i:i+bs]),return_tensors='pt',padding=True,truncation=True,max_length=256).to(GEN_DEV)
        hs=model(**enc,output_hidden_states=True).hidden_states
        m=enc.attention_mask.unsqueeze(-1)
        pooled=[((h*m).sum(1)/m.sum(1)).float().cpu().numpy() for h in hs]   # mean-pool over tokens
        acc=[[p] for p in pooled] if acc is None else [acc[l]+[pooled[l]] for l in range(len(pooled))]
    return [np.concatenate(a,0) for a in acc]
@torch.no_grad()
def reps_final(queries,bs=8):
    return reps_all(queries,bs)[-1]
def _d2(A,B):
    sa=(A**2).sum(1,keepdims=True); sb=(B**2).sum(1,keepdims=True); return np.clip(sa+sb.T-2*A@B.T,0,None)
def hsic(X,Y,sigma):                                   # authors' raw estimator tr(KHLH)/(n-1)^2
    n=min(len(X),len(Y)); X,Y=X[:n],Y[:n]
    K=np.exp(-_d2(X,X)/(2*sigma**2)); L=np.exp(-_d2(Y,Y)/(2*sigma**2)); H=np.eye(n)-np.ones((n,n))/n
    return float(np.trace(K@H@L@H)/(n-1)**2)
def nhsic(X,Y,sigma):                                  # normalised HSIC in [0,1], norm-robust
    hxy=hsic(X,Y,sigma); hxx=hsic(X,X,sigma); hyy=hsic(Y,Y,sigma)
    return float(hxy/np.sqrt(hxx*hyy)) if hxx>0 and hyy>0 else 0.0
def linear_cka(X,Y):                                   # linear CKA, scale-invariant
    n=min(len(X),len(Y)); X,Y=X[:n],Y[:n]              # match HSIC's min-truncation (fair/priv sizes differ)
    Xc=X-X.mean(0,keepdims=True); Yc=Y-Y.mean(0,keepdims=True)
    num=np.linalg.norm(Yc.T@Xc)**2; den=np.linalg.norm(Xc.T@Xc)*np.linalg.norm(Yc.T@Yc)
    return float(num/den) if den>0 else 0.0
def mean_norm(X): return float(np.linalg.norm(X,axis=1).mean())
@torch.no_grad()
def eval_ppl(ids,seqlen=PPL_SEQLEN):
    ns=ids.numel()//seqlen; nlls=[]
    for i in range(ns):
        inp=ids[:,i*seqlen:(i+1)*seqlen].to(GEN_DEV); lg=model(inp).logits
        nlls.append(torch.nn.CrossEntropyLoss()(lg[:,:-1,:].reshape(-1,lg.size(-1)),inp[:,1:].reshape(-1)).float()*seqlen)
    return torch.exp(torch.stack(nlls).sum()/(ns*seqlen)).item()
def suppress_reversible(coupled):
    d=dict(model.named_parameters()); saved=[]; nz=0
    for n,idxlist in coupled.items():
        if n in d and idxlist:
            p=d[n]; idx=torch.tensor(idxlist,device=p.device); flat=p.data.view(-1)
            saved.append((n,idx,flat[idx].clone())); flat[idx]=0.0; nz+=len(idxlist)
    return saved,nz
def restore(saved):
    d=dict(model.named_parameters())
    for n,idx,vals in saved: d[n].data.view(-1)[idx]=vals
def full_pack(Lf,Lp):
    # Lf,Lp = lists of per-layer reps; final layer used for the headline stats
    Xf,Xp=Lf[-1],Lp[-1]
    return {'hsic_final':{str(s):hsic(Xf,Xp,s) for s in SIGMAS},
            'hsic_per_layer':{str(s):[hsic(Lf[l],Lp[l],s) for l in range(len(Lf))] for s in SIGMAS},
            'nhsic_final':{str(s):nhsic(Xf,Xp,s) for s in SIGMAS},
            'cka_final':linear_cka(Xf,Xp),
            'norm_fair':mean_norm(Xf),'norm_priv':mean_norm(Xp)}
print('functions ready.')

## Step 4 — load coupled indices from Step 1 (checkpoint or JSON) + provenance export

In [ ]:
CJSON=find_one(['coupled_indices_health.json'])
CKPT=find_one(['health_coupled_ckpt.pt'])
if CJSON:
    obj=json.load(open(CJSON)); coupled=obj['coupled'] if 'coupled' in obj else obj
    print('coupled indices from', CJSON)
elif CKPT:
    coupled=torch.load(CKPT, map_location='cpu')['coupled']
    print('coupled indices from checkpoint', CKPT)
    json.dump({'model':MODEL_ID,'source':os.path.basename(CKPT),
               'coupled_count':int(sum(len(v) for v in coupled.values())),
               'coupled':{k:list(map(int,v)) for k,v in coupled.items()}},
              open('/kaggle/working/coupled_indices_health.json','w'))
    print('  re-exported -> coupled_indices_health.json (clean provenance artifact)')
else:
    raise FileNotFoundError('Add Step-1 coupled indices (coupled_indices_health.json or health_coupled_ckpt.pt) as input.')
coupled={k:[int(x) for x in v] for k,v in coupled.items()}
mlp_params=int(sum(p.numel() for n,p in model.named_parameters() if p.dim()==2 and 'mlp' in n and 'weight' in n))
n_zero=int(sum(len(v) for v in coupled.values()))
print('coupled matrices', len(coupled), '| weights to zero', n_zero, '| sparsity %.2e'%(n_zero/mlp_params))

## Step 5 — baseline (no suppression), measured once

In [ ]:
print('baseline: reps + HSIC/nHSIC/CKA/norms + perplexity ...')
Lf0=reps_all(fair_pooled); Lp0=reps_all(priv_pooled)
base=full_pack(Lf0,Lp0)
Xf0,Xp0=Lf0[-1].copy(),Lp0[-1].copy()             # keep final-layer reps for the bootstrap
del Lf0,Lp0; gc.collect(); torch.cuda.empty_cache()
base['wikitext_ppl']=eval_ppl(wt_ids)
base['med_ppl']=eval_ppl(med_ids) if MED_PPL else None
# per-subset final-layer reps (baseline)
sub_reps0={s:(reps_final(fair_by_sub[s]) if fair_by_sub[s] else None) for s in SUBSET_ORDER}
priv_sub_rep0=reps_final(priv_sub)
base['hsic_per_subset']={s:({str(sg):hsic(sub_reps0[s],priv_sub_rep0,sg) for sg in SIGMAS}
                            if sub_reps0[s] is not None else None) for s in SUBSET_ORDER}
print('  HSIC(final):',{s:round(base['hsic_final'][str(s)],6) for s in SIGMAS})
print('  wikitext ppl %.4f'%base['wikitext_ppl'], '| med ppl', (round(base['med_ppl'],4) if MED_PPL else 'off'))
memline('baseline')

## Step 6 — SPIN (coupled set zeroed), same measurements, then restore

In [ ]:
saved,nz=suppress_reversible(coupled); print('zeroed', nz, 'weights')
Lf1=reps_all(fair_pooled); Lp1=reps_all(priv_pooled)
spin=full_pack(Lf1,Lp1)
Xf1,Xp1=Lf1[-1].copy(),Lp1[-1].copy()
del Lf1,Lp1; gc.collect(); torch.cuda.empty_cache()
spin['wikitext_ppl']=eval_ppl(wt_ids)
spin['med_ppl']=eval_ppl(med_ids) if MED_PPL else None
sub_reps1={s:(reps_final(fair_by_sub[s]) if fair_by_sub[s] else None) for s in SUBSET_ORDER}
priv_sub_rep1=reps_final(priv_sub)
spin['hsic_per_subset']={s:({str(sg):hsic(sub_reps1[s],priv_sub_rep1,sg) for sg in SIGMAS}
                            if sub_reps1[s] is not None else None) for s in SUBSET_ORDER}
restore(saved); print('weights restored (reversible).')

def pct(a,b): return 100*(a-b)/b if b else 0.0
print('\n=== decoupling (HSIC final, baseline -> SPIN) ===')
for s in SIGMAS:
    b=base['hsic_final'][str(s)]; sp=spin['hsic_final'][str(s)]
    print(f'  σ{s:>3}: raw {b:.6f} -> {sp:.6f} ({pct(sp,b):+.1f}%) | nHSIC {base["nhsic_final"][str(s)]:.4f} -> {spin["nhsic_final"][str(s)]:.4f}')
print(f'  linear CKA {base["cka_final"]:.4f} -> {spin["cka_final"]:.4f}')
print(f'  rep norms fair {base["norm_fair"]:.3f}->{spin["norm_fair"]:.3f} | priv {base["norm_priv"]:.3f}->{spin["norm_priv"]:.3f}')
print('\n=== capability ===')
print(f'  WikiText-2 ppl {base["wikitext_ppl"]:.4f} -> {spin["wikitext_ppl"]:.4f} ({pct(spin["wikitext_ppl"],base["wikitext_ppl"]):+.2f}%)')
if MED_PPL: print(f'  PubMedQA  ppl {base["med_ppl"]:.4f} -> {spin["med_ppl"]:.4f} ({pct(spin["med_ppl"],base["med_ppl"]):+.2f}%)')
print('\n=== per-subset HSIC (final, σ=200; exploratory — small subsets noisy) ===')
for s in SUBSET_ORDER:
    b=base['hsic_per_subset'][s]; sp=spin['hsic_per_subset'][s]
    if b is None: print(f'  {s:<12} (no questions)'); continue
    print(f'  {s:<12} n={len(fair_by_sub[s]):<4} raw {b["200"]:.6f} -> {sp["200"]:.6f} ({pct(sp["200"],b["200"]):+.1f}%)')

## Step 6b — medical capability benchmarks (multiple-choice accuracy, baseline vs suppressed)

Accuracy on **MedMCQA** (1,000 seeded) and the **MMLU clinical subsets** (clinical_knowledge,
professional_medicine, college_medicine, anatomy, medical_genetics; full). Scored by length-normalised
log-likelihood of each option (pick argmax) — no judge, no external harness, so this is portable to any
GPU outside Kaggle. Complements perplexity (fluency) with correctness (does the model still answer
medical questions right after suppression). Datasets pull from Hugging Face at runtime.

In [ ]:
MMLU_CLIN=['clinical_knowledge','professional_medicine','college_medicine','anatomy','medical_genetics']
N_MEDMCQA=1000

@torch.no_grad()
def _choice_loglik(prompt, choice):
    enc=tokenizer(prompt+choice, return_tensors='pt', truncation=True, max_length=1024).to(GEN_DEV)
    plen=tokenizer(prompt, return_tensors='pt').input_ids.shape[1]
    logits=model(**enc).logits[:, :-1, :]; tgt=enc.input_ids[:, 1:]
    logp=torch.log_softmax(logits, dim=-1).gather(2, tgt.unsqueeze(-1)).squeeze(-1)[0]
    seg=logp[plen-1:]                                    # tokens belonging to the choice
    return float(seg.mean().item()) if seg.numel()>0 else -1e9
def mc_accuracy(items):
    if not items: return float('nan')
    c=0
    for it in items:
        lls=[_choice_loglik(it['prompt'], ch) for ch in it['choices']]
        c+= int(np.argmax(lls))==it['answer']
    return c/len(items)
def _load(name, subj=None):
    if name=='medmcqa':
        try: ds=load_dataset('openlifescienceai/medmcqa', split='validation')
        except Exception: ds=load_dataset('medmcqa', split='validation', trust_remote_code=True)
        ds=ds.shuffle(seed=SEED).select(range(min(N_MEDMCQA,len(ds)))); out=[]
        for r in ds:
            out.append({'prompt':f"Question: {r['question']}\nAnswer:",
                        'choices':[f" {o}" for o in (r['opa'],r['opb'],r['opc'],r['opd'])],
                        'answer':int(r['cop'])})
        return out
    try: ds=load_dataset('cais/mmlu', subj, split='test')
    except Exception: ds=load_dataset('cais/mmlu', subj, split='test', trust_remote_code=True)
    return [{'prompt':f"Question: {r['question']}\nAnswer:",
             'choices':[f" {c}" for c in r['choices']],'answer':int(r['answer'])} for r in ds]

BENCH={'MedMCQA':_load('medmcqa')}
for s in MMLU_CLIN: BENCH['MMLU:'+s]=_load('mmlu', s)
print('loaded:', {k:len(v) for k,v in BENCH.items()})

def score_all(): return {k:mc_accuracy(v) for k,v in BENCH.items()}
print('scoring baseline accuracy...'); acc_base=score_all()
saved,_=suppress_reversible(coupled)
print('scoring suppressed accuracy...'); acc_spin=score_all(); restore(saved)
def _pooled(a):
    tot=sum(len(BENCH['MMLU:'+s]) for s in MMLU_CLIN)
    return sum(a['MMLU:'+s]*len(BENCH['MMLU:'+s]) for s in MMLU_CLIN)/tot if tot else float('nan')
bench_results={'per_benchmark':{k:{'base':acc_base[k],'spin':acc_spin[k],'n':len(BENCH[k])} for k in BENCH},
               'mmlu_clinical_pooled':{'base':_pooled(acc_base),'spin':_pooled(acc_spin)}}
print('\n=== medical capability (accuracy, baseline -> SPIN) ===')
for k in BENCH:
    b,s=acc_base[k],acc_spin[k]; print(f'  {k:<26} n={len(BENCH[k]):<4} {b:.3f} -> {s:.3f} ({100*(s-b):+.1f} pp)')
pp=bench_results['mmlu_clinical_pooled']
print(f"  {'MMLU-clinical (pooled)':<26}      {pp['base']:.3f} -> {pp['spin']:.3f} ({100*(pp['spin']-pp['base']):+.1f} pp)")

## Step 7 — bootstrap 95% CIs on the HSIC drop (paired resampling), per σ

In [ ]:
boot={}
if BOOT:
    n0=min(len(Xf0),len(Xp0)); rng=np.random.default_rng(SEED)
    IDX=[rng.integers(0,n0,n0) for _ in range(BOOT)]
    def _K(X,sig): return np.exp(-_d2(X[:n0],X[:n0])/(2*sig**2))
    def _hsic_idx(K,L,idx):
        Kb=K[np.ix_(idx,idx)]; Lb=L[np.ix_(idx,idx)]; n=len(idx)
        Kc=Kb-Kb.mean(0,keepdims=True)-Kb.mean(1,keepdims=True)+Kb.mean()
        return (Kc*Lb).sum()/(n-1)**2
    for s in SIGMAS:
        Kf0,Lp0k=_K(Xf0,s),_K(Xp0,s); Kf1,Lp1k=_K(Xf1,s),_K(Xp1,s); arr=np.empty(BOOT)
        for b,idx in enumerate(IDX):
            hb=_hsic_idx(Kf0,Lp0k,idx); hs=_hsic_idx(Kf1,Lp1k,idx)
            arr[b]=100*(hs-hb)/hb if hb else 0.0
        boot[str(s)]={'drop_med':float(np.median(arr)),'lo':float(np.percentile(arr,2.5)),
                      'hi':float(np.percentile(arr,97.5)),
                      'significant':bool(np.percentile(arr,2.5)<0 and np.percentile(arr,97.5)<0)}
    print('HSIC drop %, median [95% CI]  (significant = CI entirely below 0):')
    for s in SIGMAS:
        d=boot[str(s)]; print(f'  σ{s:>3}: {d["drop_med"]:+6.1f}%  [{d["lo"]:+6.1f}, {d["hi"]:+6.1f}]  {"SIG" if d["significant"] else "n.s."}')
else:
    print('BOOT=0: skipped.')

## Step 8 — assemble `metrics_health_step2.json`

In [ ]:
def deltas(a,b,keys):
    return {k:{'base':a[k] if not isinstance(a[k],dict) else a[k],
               'spin':b[k] if not isinstance(b[k],dict) else b[k]} for k in keys}
out={'model':MODEL_ID,'seed':SEED,'sigmas':SIGMAS,'use_chat_template':USE_CHAT_TEMPLATE,
     'eval':{'fair_pooled':len(fair_pooled),'priv_pooled':len(priv_pooled),
             'fair_heldout_total':int(len(eval_fair)),'priv_eval_total':int(len(eval_priv)),
             'fair_per_subset_heldout':{s:int((eval_fair['subset']==s).sum()) for s in SUBSET_ORDER}},
     'suppression':{'weights_zeroed':int(nz),'mlp_params':mlp_params,'sparsity':nz/mlp_params},
     'baseline':base,'spin':spin,
     'delta':{'hsic_final_pct':{str(s):pct(spin['hsic_final'][str(s)],base['hsic_final'][str(s)]) for s in SIGMAS},
              'wikitext_ppl_pct':pct(spin['wikitext_ppl'],base['wikitext_ppl']),
              'med_ppl_pct':(pct(spin['med_ppl'],base['med_ppl']) if MED_PPL else None),
              'hsic_per_subset_pct':{s:(pct(spin['hsic_per_subset'][s]['200'],base['hsic_per_subset'][s]['200'])
                                        if base['hsic_per_subset'][s] else None) for s in SUBSET_ORDER}},
     'bootstrap':{'B':BOOT,'drop_ci':boot},
     'capability_benchmarks':bench_results}
json.dump(out, open('/kaggle/working/metrics_health_step2.json','w'), indent=1)
print('saved metrics_health_step2.json')
print('artifacts: metrics_health_step2.json, eval_fair_health.csv, eval_priv_health.csv, coupled_indices_health.json')